# Boosted Decision Tree

GBDT costruisce alberi in modo sequenziale, correggendo progressivamente gli errori di classificazione del modello costruito fino a quel punto, producendo un modello complesso e molto accurato

In [34]:
import pandas as pd
import numpy as np

from joblib import Parallel, delayed

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from sklearn.multioutput import MultiOutputClassifier

from pathlib import Path
import warnings
# Nascondo i warning
warnings.filterwarnings('ignore')
# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Lavoro su singola fold

In [ ]:
def fit_single_fold(train_idx, test_idx, features, target, groups, max_iter, max_depth, min_samples_leaf):


    # ========== DEBUGGING: Stampo indici train/test  ==========
    
    """print("?"*50 + "\nDebug\n" + "?"*50)
    print(f"\nFold {fold} - File: {csv_name}")        
    print(f"  Train indice: {train_index[:10]})")
    print(f"  Test indice: {test_index[:10]})")
    print(f"  Train gruppo (Patient IDs): {groups.iloc[train_index].unique()}")
    print(f"  Test gruppo (Patient IDs): {groups.iloc[test_index].unique()}")
    print("?"*100)"""
    # ==========================================================

    X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
    y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]

    base_clf = HistGradientBoostingClassifier(
        max_iter=max_iter,
        max_depth=max_depth,
        min_samples_leaf=min_samples_leaf,
        random_state=42
    )
    rf = MultiOutputClassifier(base_clf, n_jobs=-1)

    # Addestro il modello sul train set di questa fold
    rf.fit(X_train, y_train)
    
    # Predico il target sul test set di questa fold
    y_pred = rf.predict(X_test)
    
    # Serve per debug 
    #score = f1_score(y_test, y_pred, average="micro")
    #print(f"Fold {fold} - {max_iter=}, {max_depth=}, {min_samples_leaf=}, Score={score:.4f}")

    return f1_score(y_test, y_pred, average="micro")

# Training


In [ ]:
def training(file_path, csv_name):
    # Leggo i csv
    df = pd.read_csv(file_path)

    # Mi definisco la lista dei target
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]']

    # Filtro solo le pazienti con PR valido
    df_validi = df.dropna(subset=original_target_list).copy()

    # Trasformo tutto in valori binari per "facilitare" il lavoro
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)
    
    # Lista finale delle colonne target binarizzate che verranno usate per l'addestramento
    final_target_list = ['PR_class', 'ER_class', 'KI67_class']
    
    """ 
        Preparo le feature (X) e i target (y) per il modello
    """
    # Definisco tutte le colonne da rimuovere per ottenere solo le feature radiomiche
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    # 'target' contiene le 3 colonne da usare
    target = df_validi[final_target_list]
    
    # 'groups' contiene l'ID del paziente per ogni lesione.
    # Mi serve per fare la cross-validation a gruppo
    groups = df_validi['Patient ID']

    # Riempip a Nan se è rimasto vuoto
    features = features.fillna(features.mean())


    # Imposto la strategia di cross-validation.
    # GroupKFold assicura che le lesioni dello stesso paziente non vengano mai divise tra training set e test set
    cv = GroupKFold(n_splits=5, shuffle=True, random_state=42)
    


    # Definisco gli iperparametri
    iperparametri = {
        'max_iter': [100, 200],       # numero di iterazioni boosting
        'max_depth': [10, 20, None],  # profondità massima degli alberi
        'min_samples_leaf': [1, 2]    # min campioni in una foglia
    }


    # Lista vuota per collezionare i punteggi di performance di ogni fold.
    scores = []

    # ciclo sui valori massimi delle iterazioni
    for max_iter in iperparametri['max_iter']:  
        # ciclo sui valori della profondità massima degli alberi
        for max_depth in iperparametri['max_depth']:
            # ciclo sui diversi valori per il minimo numero di campioni in una foglia
            for min_samples_leaf in iperparametri['min_samples_leaf']:
                # Parallelizzo i fold della cross-validation
                fold_scores = Parallel(n_jobs=-1)( delayed(fit_single_fold)(train_idx, test_idx, features, target, groups, max_iter, max_depth, min_samples_leaf) for train_idx, test_idx in cv.split(features, target, groups))

                # calcolo media e deviazione standard degli score su tutte le fold
                mean_score = np.mean(fold_scores)
                std_score = np.std(fold_scores)

                # registro i risultati per la combinazione di parametri corrente
                scores.append({
                    'max_iter': max_iter,
                    'max_depth': max_depth,
                    'min_samples_leaf': min_samples_leaf,
                    'mean_score': mean_score,
                    'std_score': std_score,
                    'fold_scores': fold_scores
                })


                        
    return scores

# Lettura dei file

In [37]:
results_per_dataset = {}

for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Stampo i risultati per ogni dataset
for name, metrics_list in results_per_dataset.items():
    
    # Stampo solo i migliori
    best_result = max(metrics_list, key=lambda x: x['mean_score'])
    print(f"\nDataset: {name}:")
    print(f"  max_iter: {best_result['max_iter']}")
    print(f"  max_depth: {best_result['max_depth']}")
    print(f"  min_samples_leaf: {best_result['min_samples_leaf']}")
    print(f"  Mean F1-score: {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}\n")



# va dentro al for sopra



Dataset: t2_medsam:
  max_iter: 100
  max_depth: 10
  min_samples_leaf: 2
  Mean F1-score: 0.683 ± 0.079


Dataset: t2_preprocessed:
  max_iter: 200
  max_depth: 10
  min_samples_leaf: 2
  Mean F1-score: 0.677 ± 0.105


Dataset: t2_original:
  max_iter: 100
  max_depth: 10
  min_samples_leaf: 1
  Mean F1-score: 0.697 ± 0.075


Dataset: medsam_dynamic:
  max_iter: 100
  max_depth: 10
  min_samples_leaf: 1
  Mean F1-score: 0.687 ± 0.086


Dataset: preprocessed_dynamic:
  max_iter: 200
  max_depth: 10
  min_samples_leaf: 2
  Mean F1-score: 0.708 ± 0.091


Dataset: original_dynamic:
  max_iter: 100
  max_depth: 10
  min_samples_leaf: 2
  Mean F1-score: 0.689 ± 0.037

